# 02 · BGE-M3: embeddings densos e índice exacto

**TESIS-ISBD · BGE_M3_DENSO_RAG_V1**

Segundo codificador de la comparación: **`BAAI/bge-m3`**, exclusivamente en modalidad densa.

BGE-M3 no necesita la instrucción `query` de Qwen. Documentos y consultas se codifican sin prefijo. No se calculan métricas de calidad ni se aprueban consultas pendientes.

## 1. Verificar el catálogo y las rutas

Leer desde `06_grafos/catalogo_articulos.csv` y `07_RAG/auditoria_catalogo.json`.
La carpeta de trabajo puede ser `07_RAG` o la raíz del repositorio.
Esta celda no descarga ni ejecuta modelos.

In [1]:
import csv
import hashlib
import json
import sys
from pathlib import Path
from collections import Counter

VERSION = "BGE_M3_DENSO_RAG_V1"
MODELO = "BAAI/bge-m3"
# Commit completo de Hugging Face: se envía a la carga y se guarda en el NPZ.
REVISION = "5617a9f61b028005a4858fdac845db406aefb181"
LOTE = 4
DIMENSION_ESPERADA = 1024
SHA_CATALOGO_ESPERADO = "74040d185b762721c29521fdc3a270740569f009b7ae40dbce654366b576598b"

def sha(ruta):
    return hashlib.sha256(ruta.read_bytes()).hexdigest()

REPO = next((p.resolve() for p in (Path.cwd(), Path.cwd().parent)
             if (p / "06_grafos" / "catalogo_articulos.csv").is_file()
             and (p / "07_RAG" / "auditoria_catalogo.json").is_file()), None)
if REPO is None:
    raise FileNotFoundError("Abre esta libreta desde 07_RAG o Tesis_Multimodelo.")

BASE = REPO / "07_RAG"
RUTA_CATALOGO = REPO / "06_grafos" / "catalogo_articulos.csv"
RUTA_AUDITORIA = BASE / "auditoria_catalogo.json"
RUTA_CONSULTAS = BASE / "consultas_para_revision.csv"
VECTORES = BASE / "bge_m3_vectores_v1.npz"
INDICE = BASE / "bge_m3_indice_v1.faiss"

auditoria = json.loads(RUTA_AUDITORIA.read_text(encoding="utf-8"))
SHA_CATALOGO = sha(RUTA_CATALOGO)
if not (SHA_CATALOGO == SHA_CATALOGO_ESPERADO
        == auditoria["fuentes"]["catalogo"]["sha256_bytes"]):
    raise ValueError("El catálogo no corresponde a la versión auditada y usada con Qwen.")

with RUTA_CATALOGO.open(encoding="utf-8-sig", newline="") as f:
    articulos = list(csv.DictReader(f))
article_ids = [r["article_id"] for r in articulos]
resumenes = [r["abstract"] for r in articulos]
if (len(article_ids) != 519 or len(set(article_ids)) != 519
        or len(set(resumenes)) != 514 or not all(t.strip() for t in resumenes)):
    raise ValueError("Cambió la estructura del catálogo o hay resúmenes vacíos.")

consultas = []
if RUTA_CONSULTAS.is_file():
    with RUTA_CONSULTAS.open(encoding="utf-8-sig", newline="") as f:
        consultas = list(csv.DictReader(f))
    print("Consultas:", len(consultas), "| Revisión humana:",
          dict(Counter(c.get("revision_humana", "no registrada") for c in consultas)))

print("Python:", sys.version.split()[0])
print("Catálogo verificado, SHA-256:", SHA_CATALOGO)
print("Artículos:", len(article_ids), "| Resúmenes distintos:", len(set(resumenes)))
print("Modelo:", MODELO, "| Revisión fijada:", REVISION)
print("Salidas:", VECTORES.name, "y", INDICE.name, "dentro de", BASE)
print("Esta celda NO ejecuta ningún modelo.")

Consultas: 24 | Revisión humana: {'pendiente': 24}
Python: 3.11.16
Catálogo verificado, SHA-256: 74040d185b762721c29521fdc3a270740569f009b7ae40dbce654366b576598b
Artículos: 519 | Resúmenes distintos: 514
Modelo: BAAI/bge-m3 | Revisión fijada: 5617a9f61b028005a4858fdac845db406aefb181
Salidas: bge_m3_vectores_v1.npz y bge_m3_indice_v1.faiss dentro de C:\Users\hazar\Documents\GitHub\Tesis_Multimodelo\07_RAG
Esta celda NO ejecuta ningún modelo.


## 2. Comprobar bibliotecas y preparar la normalización

Se utilizan las mismas bibliotecas instaladas para Qwen.

La precisión de los pesos se deja en la configuración del modelo y se registra después de cargarlo.

In [2]:
try:
    import numpy as np
    import torch
    import faiss
    import sentence_transformers
    import transformers
    import huggingface_hub
    from sentence_transformers import SentenceTransformer
except ImportError as exc:
    raise ImportError("Falta una biblioteca en este kernel. Usa el entorno donde funcionó Qwen.") from exc

print("numpy:", np.__version__)
print("torch:", torch.__version__, "| CUDA disponible:", torch.cuda.is_available())
print("sentence-transformers:", sentence_transformers.__version__)
print("transformers:", transformers.__version__)
print("faiss:", getattr(faiss, "__version__", "no declarada"))
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("AVISO: se usará CPU. No se ocultará esta diferencia en el registro.")

def sincronizar():
    # Evitar terminar el cronómetro mientras queda trabajo de CUDA pendiente.
    if torch.cuda.is_available():
        torch.cuda.synchronize()

def normalizar_l2_float32(salida, filas_esperadas, dimension_esperada):
    v = np.array(salida, dtype=np.float32, order="C", copy=True)
    if v.shape != (filas_esperadas, dimension_esperada) or not np.isfinite(v).all():
        raise ValueError("Forma o valores de embeddings incorrectos.")
    antes = np.linalg.norm(v, axis=1)
    if not np.isfinite(antes).all() or np.any(antes <= 0):
        raise ValueError("Vector nulo o norma no finita; no se guarda ningún índice.")
    faiss.normalize_L2(v)
    despues = np.linalg.norm(v, axis=1)
    if not np.isfinite(v).all() or not np.allclose(despues, 1, rtol=0, atol=1e-5):
        raise ValueError("Falló la normalización L2 en float32.")
    return v, antes, despues

print("Bibliotecas listas. Todavía no se ha cargado BGE-M3.")

W1107 18:39:09.220000 24724 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


numpy: 2.3.5
torch: 2.11.0+cu128 | CUDA disponible: True
sentence-transformers: 6.2.0
transformers: 5.22.2
faiss: 1.15.1
GPU: NVIDIA GeForce RTX 4070 SUPER
Bibliotecas listas. Todavía no se ha cargado BGE-M3.


## 3. Generar los vectores y guardar el índice

Cambiar **únicamente `EJECUTAR = True`** después de comprobar las dos celdas anteriores.

In [3]:
EJECUTAR = True

if not EJECUTAR:
    print("Preparado. No se ha descargado BGE-M3 ni creado su índice.")
else:
    if VECTORES.exists() or INDICE.exists():
        raise FileExistsError("Ya existe una salida de BGE-M3. No se sobrescribe; revisa antes de repetir.")

    from time import perf_counter
    from datetime import datetime, timezone

    dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
    sincronizar()
    inicio = perf_counter()
    codificador_bge = SentenceTransformer(MODELO, revision=REVISION, device=dispositivo)
    codificador_bge.eval()
    # Sin instrucciones de Qwen ni prefijos de consulta/documento en BGE-M3.
    if getattr(codificador_bge, "default_prompt_name", None) is not None:
        raise ValueError("Se detectó un prefijo predeterminado inesperado; revisar antes de codificar.")
    sincronizar()
    t_carga = perf_counter() - inicio
    dtype_modelo = str(next(codificador_bge.parameters()).dtype)
    print("Modelo cargado en", dispositivo, "| Carga:", round(t_carga, 3), "s")
    print("Precisión del modelo:", dtype_modelo, "| Revisión:", REVISION)

    inicio = perf_counter()
    limite_tokens = int(codificador_bge.max_seq_length)
    if limite_tokens <= 0 or limite_tokens > 8192:
        raise ValueError("Límite de entrada inesperado para el perfil BGE-M3.")
    tokenizados = codificador_bge.tokenizer(
        resumenes, add_special_tokens=True, padding=False, truncation=False
    )
    longitudes = np.asarray([len(x) for x in tokenizados["input_ids"]], dtype=np.int64)
    del tokenizados
    excedidos = [article_ids[i] for i in np.flatnonzero(longitudes > limite_tokens)]
    print("Tokens por resumen (mín./máx.):", int(longitudes.min()), int(longitudes.max()),
          "| Límite efectivo:", limite_tokens)
    if excedidos:
        raise ValueError("Resúmenes que exceden el límite; no se truncaron: " + ", ".join(excedidos))
    t_longitudes = perf_counter() - inicio

    sincronizar()
    inicio = perf_counter()
    salida_encode = codificador_bge.encode(
        resumenes, batch_size=LOTE, show_progress_bar=True,
        normalize_embeddings=True, convert_to_numpy=True
    )
    sincronizar()
    t_vectorizacion = perf_counter() - inicio
    dtype_salida_encode = str(np.asarray(salida_encode).dtype)
    inicio = perf_counter()
    vectores, normas_antes, normas_despues = normalizar_l2_float32(
        salida_encode, len(article_ids), DIMENSION_ESPERADA
    )
    t_normalizacion = perf_counter() - inicio
    del salida_encode
    print("Salida de encode:", dtype_salida_encode)
    print("Normas ANTES (mín./máx.):", float(normas_antes.min()), float(normas_antes.max()))
    print("Normas DESPUÉS (mín./máx.):", float(normas_despues.min()), float(normas_despues.max()))
    print("Desviación máxima DESPUÉS:", float(np.max(np.abs(normas_despues - 1))))

    inicio = perf_counter()
    indice = faiss.IndexFlatIP(DIMENSION_ESPERADA)
    indice.add(vectores)
    t_indice = perf_counter() - inicio
    if indice.ntotal != len(article_ids):
        raise ValueError("El índice no contiene todos los artículos.")

    # Solo se escriben estas dos salidas; todos los metadatos acompañan al NPZ.
    with VECTORES.open("xb") as f:
        np.savez_compressed(
            f, vectores=vectores, article_ids=np.asarray(article_ids),
            modelo=np.asarray(MODELO), modelo_revision=np.asarray(REVISION),
            revision_origen=np.asarray("Commit explícito enviado a SentenceTransformer"),
            version_codigo=np.asarray(VERSION), modo=np.asarray("dense"),
            sha256_catalogo=np.asarray(SHA_CATALOGO), sha256_auditoria=np.asarray(sha(RUTA_AUDITORIA)),
            fecha_utc=np.asarray(datetime.now(timezone.utc).isoformat()),
            dispositivo=np.asarray(dispositivo), dtype_modelo=np.asarray(dtype_modelo),
            dtype_salida_encode=np.asarray(dtype_salida_encode),
            campo=np.asarray("abstract"), batch_size=np.asarray(LOTE),
            prefijo_documento=np.asarray(""), prefijo_consulta=np.asarray(""),
            max_seq_length=np.asarray(limite_tokens), tokens_por_articulo=longitudes,
            articulos_truncados=np.asarray(0),
            normalizacion_final=np.asarray("faiss.normalize_L2 sobre float32; rtol=0; atol=1e-5"),
            normas_antes_min=np.asarray(float(normas_antes.min())),
            normas_antes_max=np.asarray(float(normas_antes.max())),
            desviacion_norma_antes_max=np.asarray(float(np.max(np.abs(normas_antes - 1)))),
            normas_despues_min=np.asarray(float(normas_despues.min())),
            normas_despues_max=np.asarray(float(normas_despues.max())),
            desviacion_norma_despues_max=np.asarray(float(np.max(np.abs(normas_despues - 1)))),
            tiempo_carga_s=np.asarray(t_carga), tiempo_verificacion_longitud_s=np.asarray(t_longitudes),
            tiempo_vectorizacion_s=np.asarray(t_vectorizacion), tiempo_normalizacion_s=np.asarray(t_normalizacion),
            tiempo_construccion_indice_s=np.asarray(t_indice),
            python=np.asarray(sys.version.split()[0]), numpy_version=np.asarray(np.__version__),
            torch_version=np.asarray(torch.__version__),
            sentence_transformers_version=np.asarray(sentence_transformers.__version__),
            transformers_version=np.asarray(transformers.__version__),
            huggingface_hub_version=np.asarray(huggingface_hub.__version__),
            faiss_version=np.asarray(getattr(faiss, "__version__", "no_registrada")),
        )
    if INDICE.exists():
        raise FileExistsError("El índice apareció durante la ejecución. No se reemplaza.")
    faiss.write_index(indice, str(INDICE))

    # Comprobar también la matriz interna del índice, no solo su tamaño.
    with np.load(VECTORES, allow_pickle=False) as guardado:
        if (guardado["article_ids"].tolist() != article_ids
                or guardado["sha256_catalogo"].item() != SHA_CATALOGO
                or not np.array_equal(guardado["vectores"], vectores)):
            raise ValueError("La relectura del NPZ no coincide con lo generado.")
    indice_guardado = faiss.read_index(str(INDICE))
    if (indice_guardado.ntotal != len(article_ids) or indice_guardado.d != DIMENSION_ESPERADA
            or not np.array_equal(indice_guardado.reconstruct_n(0, len(article_ids)), vectores)):
        raise ValueError("El índice guardado no coincide con los vectores.")

    print("Artículos indexados:", indice_guardado.ntotal, "| Dimensión:", indice_guardado.d,
          "| Vectorización:", round(t_vectorizacion, 3), "s")
    print("Guardados:", VECTORES.name, "y", INDICE.name)
    print("SHA-256 NPZ:", sha(VECTORES))
    print("SHA-256 FAISS:", sha(INDICE))
    print("No se han calculado métricas de recuperación ni modificado resultados de Qwen.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\hazar\anaconda3\envs\entorno_tesis\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\hazar\.cache\huggingface\hub\models--BAAI--bge-m3. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

... (12 outputs left out)

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Modelo cargado en cuda | Carga: 49.347 s
Precisión del modelo: torch.float32 | Revisión: 5617a9f61b028005a4858fdac845db406aefb181
Tokens por resumen (mín./máx.): 40 703 | Límite efectivo: 8192


Batches:   0%|          | 0/130 [00:00<?, ?it/s]

Salida de encode: float32
Normas ANTES (mín./máx.): 0.9999999403953552 1.0000001192092896
Normas DESPUÉS (mín./máx.): 0.9999998807907104 1.0000001192092896
Desviación máxima DESPUÉS: 1.1920928955078125e-07
Artículos indexados: 519 | Dimensión: 1024 | Vectorización: 10.4 s
Guardados: bge_m3_vectores_v1.npz y bge_m3_indice_v1.faiss
SHA-256 NPZ: 5c0b99c8383111e1eb5bca9a38fab3b539d3b27cf64c87a886413ff3793b4feb
SHA-256 FAISS: b26b28a97d0b4b37f0c58a23d5c9e1d9859e80414f0ac743d47e3cd596c4718b
No se han calculado métricas de recuperación ni modificado resultados de Qwen.


## 4. Búsqueda ilustrativa, sin calificación de pertinencia

Se consulta el índice de BGE con un vector de consulta de BGE. **Aunque ambos produzcan 1024 dimensiones, no se deben usar vectores de Qwen en el índice de BGE ni viceversa.**

In [4]:
if not VECTORES.is_file() or not INDICE.is_file():
    print("Primero completa la tercera celda con EJECUTAR=True.")
elif "codificador_bge" not in globals():
    print("Los resultados están guardados. Para la demostración, carga solo BGE, sin repetir la vectorización:")
    print('codificador_bge = SentenceTransformer(MODELO, revision=REVISION, device="cuda" if torch.cuda.is_available() else "cpu")')
else:
    with np.load(VECTORES, allow_pickle=False) as guardado:
        ids_guardados = guardado["article_ids"].tolist()
        if (ids_guardados != article_ids or guardado["sha256_catalogo"].item() != SHA_CATALOGO
                or guardado["modelo"].item() != MODELO
                or guardado["modelo_revision"].item() != REVISION):
            raise ValueError("Los vectores no corresponden a este catálogo y perfil BGE.")
        matriz_guardada = guardado["vectores"].copy()
    indice_busqueda = faiss.read_index(str(INDICE))
    if (indice_busqueda.ntotal != len(ids_guardados) or indice_busqueda.d != DIMENSION_ESPERADA
            or not np.array_equal(indice_busqueda.reconstruct_n(0, len(ids_guardados)), matriz_guardada)):
        raise ValueError("El índice no coincide con el NPZ.")
    del matriz_guardada

    q = (consultas[0]["consulta_propuesta"] if consultas else
         "¿Qué artículos estudian documentación técnica mediante RAG?")
    tokens_q = codificador_bge.tokenizer(q, add_special_tokens=True, truncation=False)["input_ids"]
    if len(tokens_q) > int(codificador_bge.max_seq_length):
        raise ValueError("La consulta excede el límite de entrada; no se trunca.")
    salida_q = codificador_bge.encode([q], normalize_embeddings=True, convert_to_numpy=True)
    vq, _, _ = normalizar_l2_float32(salida_q, 1, indice_busqueda.d)
    _, posiciones = indice_busqueda.search(vq, min(5, indice_busqueda.ntotal))
    por_id = {r["article_id"]: r for r in articulos}
    print("Consulta de ejemplo:", q)
    print("Cinco resultados de BGE-M3, SIN calificación de pertinencia:")
    for posicion in posiciones[0].tolist():
        if posicion >= 0:
            aid = ids_guardados[posicion]
            print(f"article_id={aid} | {por_id[aid]['titulo'][:95]}")

Consulta de ejemplo: ¿Qué trabajos permiten consultar documentación técnica de un sistema mediante preguntas en lenguaje natural y fundamentan la respuesta en material recuperado?
Cinco resultados de BGE-M3, SIN calificación de pertinencia:
article_id=117 | A retrieval-augmented-generation pipeline to help users query system-provided documentation
article_id=285 | A Systematic Literature Review of 10 years of Research on Program Synthesis and Natural Languag
article_id=325 | The Bible as a Corpus for Language Technologies
article_id=146 | Comparative Analysis of Information Extraction Techniques in the Mexican Legal System
article_id=289 | BIOMX-DB: A web application for the BIOFACQUIM natural product database
